<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 220px; height: 150px; vertical-align: middle;">
            <img src="../assets/aaa.png" width="220" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Autonomous Traders</h2>
            <span style="color:#ff7800;">An equity trading simulation to illustrate autonomous agents powered by tools and resources from MCP servers.
            </span>
        </td>
    </tr>
</table>

### Week 6 Day 4

And now - introducing the Capstone project:


# Autonomous Traders

An equity trading simulation, with 4 Traders and a Researcher, powered by a slew of MCP servers with tools & resources:

1. Our home-made Accounts MCP server (written by our engineering team!)
2. Fetch (get webpage via a local headless browser)
3. Memory
4. Brave Search
5. Financial data

And a resource to read information about the trader's account, and their investment strategy.

The goal of today's lab is to make a new python module, `traders.py` that will manage a single trader on our trading floor.

We will experiment and explore in the lab, and then migrate to a python module when we're ready.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">One more time --</h2>
            <span style="color:#ff7800;">Please do not use this for actual trading decisions!!
            </span>
        </td>
    </tr>
</table>

In [5]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account
from simple_agent import agent

load_dotenv(override=True)

True

### Let's start by gathering the MCP params for our trader

In [7]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
polygon_plan = os.getenv("POLYGON_PLAN")

is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

print(f"paid = {is_paid_polygon}")
print(f"realtime = {is_realtime_polygon}")

paid = False
realtime = False


In [5]:
if is_paid_polygon or is_realtime_polygon:
    market_mcp = {"command": "uvx","args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@master", "mcp_polygon"], "env": {"POLYGON_API_KEY": polygon_api_key}}
else:
    market_mcp = ({"command": "uv", "args": ["run", "market_server.py"]})

trader_mcp_server_params = [
    {"command": "uv", "args": ["run", "accounts_server.py"]},
    # {"command": "uv", "args": ["run", "push_server.py"]},
    market_mcp
]

### And now for our researcher

In [6]:
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

researcher_mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env}
]

### Now create the MCPServerStdio for each

In [7]:
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in researcher_mcp_server_params]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

### Now let's make a Researcher Agent to do market research

And turn it into a tool - remember how this works for OpenAI Agents SDK, and the difference with handoffs?

In [10]:
async def get_researcher(mcp_servers) -> Agent:
    instructions = f"""You are a financial researcher. You are able to search the web for interesting financial news,
look for possible trading opportunities, and help with research.
Based on the request, you carry out necessary research and respond with your findings.
Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
If there isn't a specific request, then just respond with investment opportunities based on searching latest news.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    researcher = agent(
        name="Researcher",
        instructions=instructions,
        model="deepseek-chat",
        mcp_servers=mcp_servers,
    )
    return researcher

In [11]:
async def get_researcher_tool(mcp_servers) -> Tool:
    researcher = await get_researcher(mcp_servers)
    return researcher.as_tool(
            tool_name="Researcher",
            tool_description="This tool researches online for news and opportunities, \
                either based on your specific request to look into a certain stock, \
                or generally for notable financial news and opportunities. \
                Describe what kind of research you're looking for."
        )

In [12]:
research_question = "What's the latest news on Amazon?"

for server in researcher_mcp_servers:
    await server.connect()
researcher = await get_researcher(researcher_mcp_servers)
with trace("Researcher"):
    result = await Runner.run(researcher, research_question, max_turns=30)
display(Markdown(result.final_output))



Based on my comprehensive search, here's the latest news on Amazon as of February 2026:

## **Key Upcoming Event: Q4 2025 Earnings Report**
- **Date**: **February 5, 2026** (tomorrow, after market close)
- **Expected EPS**: $1.97
- **Expected Revenue**: $211.4 billion (would mark first time surpassing $700 billion annual revenue)
- **Year-over-year growth**: 10-13% expected

## **Stock Performance & Analyst Outlook**
- **Stock Performance**: AMZN has gained about 5.3% over the past five days and is up 6.7% year-to-date in 2026
- **Price Targets**: 
  - Some analysts project $340-$370 by end of 2026, with potential upside to $400
  - Others have lowered targets to $260 (from $275) due to AI CAPEX concerns
- **Options Market**: Expecting an 8.01% move post-earnings

## **Business Developments & Strategic Focus**

### **1. AI & Cloud (AWS)**
- **AWS Growth**: Expected to grow 20% year-over-year in Q4, with full-year 2026 growth projected at 23%
- **AI Investments**: Massive CAPEX expected to exceed $150 billion in 2026 (highest among "Mag 7" companies)
- **Partnerships**: AWS partnered with German automotive supplier Aumovio for self-driving vehicle development
- **AI Initiatives**: Focus on Trainium/Neuron chips, Nova/Kira, Alexa+/Rufus, and robotics

### **2. Retail & Operations**
- **Physical Retail Expansion**: Continuing dual strategy of e-commerce and physical retail
- **Automation**: Expected to automate 50,000 manual jobs, saving $7.5 billion annually by 2026
- **Logistics**: Dismantling national logistics model for more vertical integration

### **3. New Products & Services**
- **Alexa+**: Enhanced AI assistant that can navigate websites and complete bookings autonomously
- **Agentic Commerce**: New framework expected to drive ~$400 billion potential by 2030
- **Migration Acceleration Program**: Expanded to include complete digital transformation with generative AI features

## **Market Sentiment & Analyst Views**
- **Mixed Ratings**: Some analysts have downgraded to "Hold" due to AI CAPEX concerns and weak sentiment
- **Positive Outlook**: Others remain constructive citing robust holiday trends, favorable ad checks, and beatable AWS estimates
- **Valuation Target**: Some analysts eye $3 trillion valuation as AI and AWS drive growth

## **Key Areas to Watch in Earnings Report**
1. **AWS Margins**: Facing tough comparisons but expected to show strength
2. **AI CAPEX Guidance**: Critical for understanding 2026 investment plans
3. **Advertising Growth**: Continuing as high-margin segment
4. **Retail Efficiency**: Impact of automation and logistics changes
5. **2026 Guidance**: Particularly for AWS growth and overall margin expansion

The earnings report tomorrow will be crucial in determining whether Amazon's massive AI investments are translating into financial performance or if concerns about rising costs will pressure the stock.

### Look at the trace

https://platform.openai.com/traces

In [14]:
faze_initial_strategy = "You are a day trader that aggressively buys and sells shares based on news and market conditions."
Account.get("Faze").reset(faze_initial_strategy)

display(Markdown(await read_accounts_resource("Faze")))
display(Markdown(await read_strategy_resource("Faze")))

{"name": "faze", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-02-04 03:33:29", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}

You are a day trader that aggressively buys and sells shares based on news and market conditions.

### And now - to create our Trader Agent

In [15]:
agent_name = "Faze"

# Using MCP Servers to read resources
account_details = await read_accounts_resource(agent_name)
strategy = await read_strategy_resource(agent_name)

instructions = f"""
You are a trader that manages a portfolio of shares. Your name is {agent_name} and your account is under your name, {agent_name}.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
{strategy}
Your current holdings and balance is:
{account_details}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so far.
Please make use of these tools to manage your portfolio. Carry out trades as you see fit; do not wait for instructions or ask for confirmation.
"""

prompt = """
Use your tools to make decisions about your portfolio.
Investigate the news and the market, make your decision, make the trades, and respond with a summary of your actions.
"""

In [16]:
print(instructions)


You are a trader that manages a portfolio of shares. Your name is Faze and your account is under your name, Faze.
You have access to tools that allow you to search the internet for company news, check stock prices, and buy and sell shares.
Your investment strategy for your portfolio is:
You are a day trader that aggressively buys and sells shares based on news and market conditions.
Your current holdings and balance is:
{"name": "faze", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2026-02-04 03:33:29", 10000.0], ["2026-02-04 03:36:17", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}
You have the tools to perform a websearch for relevant news and information.
You have tools to check stock prices.
You have tools to buy and sell shares.
You have tools to save memory of companies, research and thinking so far.


### And to run our Trader

In [17]:
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers)
trader = agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model="deepseek-chat",
)
with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

## Summary of Trading Actions

I have executed an aggressive day trading strategy based on current market news and conditions. Here's a summary of my trades:

### **Portfolio Overview:**
- **Starting Balance:** $10,000
- **Current Balance:** $911.86
- **Total Portfolio Value:** $17,351.86
- **Total Profit/Loss:** +$7,351.86 (73.5% gain)

### **Trades Executed:**

1. **PLTR (Palantir) - 50 shares @ $39.078**
   - **Rationale:** PLTR up 7% after strong Q4 earnings beat and bullish 2026 guidance. Momentum continuation play as AI winner proving monetization.

2. **AMD (Advanced Micro Devices) - 100 shares @ $37.074**
   - **Rationale:** AMD earnings after market close today. High volatility expected with options implying significant move. Pre-earnings position to capture potential gap up/down.

3. **PYPL (PayPal) - 50 shares @ $54.108**
   - **Rationale:** PYPL down 20% after earnings miss and new CEO announcement. Oversold condition with potential for dead cat bounce. Mean reversion play.

4. **IT (Gartner) - 60 shares @ $12.024**
   - **Rationale:** Gartner down 21% after weak full-year guidance. Extreme oversold condition with potential for technical bounce.

### **Current Holdings:**
- PLTR: 50 shares
- AMD: 100 shares  
- PYPL: 50 shares
- IT: 60 shares

### **Investment Strategy:**
I maintained my aggressive day trading approach, focusing on:
- **Momentum plays** (PLTR post-earnings)
- **Pre-earnings volatility** (AMD)
- **Oversold bounce opportunities** (PYPL, IT)
- **High volatility stocks** with clear catalysts

### **Market Conditions Addressed:**
- Tech sector rotation
- Earnings season volatility
- Oversold conditions in beaten-down stocks
- AI disruption narratives creating opportunities

The portfolio is now positioned to capture moves from AMD earnings tonight, potential continuation in PLTR momentum, and mean reversion in oversold stocks like PYPL and IT. I've used nearly all available capital ($9,088.14 deployed) for maximum exposure while maintaining $911.86 in cash for potential adjustments.

### Then go and look at the trace

http://platform.openai.com/traces


In [28]:
# And let's look at the results of the trading

await read_accounts_resource(agent_name)

'{"name": "faze", "balance": 911.860000000001, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"PLTR": 50, "AMD": 100, "PYPL": 50, "IT": 60}, "transactions": [{"symbol": "PLTR", "quantity": 50, "price": 39.078, "timestamp": "2026-02-04 03:46:03", "rationale": "PLTR up 7% after strong Q4 earnings beat and bullish 2026 guidance. Momentum likely to continue as AI winner proving monetization. Aggressive day trade to capture continuation move."}, {"symbol": "AMD", "quantity": 100, "price": 37.074, "timestamp": "2026-02-04 03:46:07", "rationale": "AMD earnings after market close today. High volatility expected with options implying significant move. Aggressive pre-earnings position to capture potential gap up/down. AI chip demand remains strong narrative."}, {"symbol": "PYPL", "quantity": 50, "price": 54.108, "timestamp": "2026-02-04 03:46:14", "rationale": "PYPL down 20% after earnings miss and new CEO announcemen

### Now it's time to review the Python module made from this:

`mcp_params.py` is where the MCP servers are specified. You'll notice I've brought in some familiar friends: memory and push notifications!

`templates.py` is where the instructions and messages are set up (i.e. the System prompts and User prompts)

`traders.py` brings it all together.

You'll notice I've done something a bit fancy with code like this:

```
async with AsyncExitStack() as stack:
    mcp_servers = [await stack.enter_async_context(MCPServerStdio(params)) for params in mcp_server_params]
```

This is just a tidy way to combine our "with" statements (known as context managers) so that we don't need to do something ugly like this:

```
async with MCPServerStdio(params=params1) as mcp_server1:
    async with MCPServerStdio(params=params2) as mcp_server2:
        async with MCPServerStdio(params=params3) as mcp_server3:
            mcp_servers = [mcp_server1, mcp_server2, mcp_server3]
```

But it's equivalent.


In [1]:
from traders import Trader


In [2]:
trader = Trader("Faze")

In [3]:
await trader.run()

In [10]:
await read_accounts_resource("Faze")

'{"name": "faze", "balance": 6301.060000000001, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {"PLTR": 50, "AMD": 100}, "transactions": [{"symbol": "PLTR", "quantity": 50, "price": 39.078, "timestamp": "2026-02-04 03:46:03", "rationale": "PLTR up 7% after strong Q4 earnings beat and bullish 2026 guidance. Momentum likely to continue as AI winner proving monetization. Aggressive day trade to capture continuation move."}, {"symbol": "AMD", "quantity": 100, "price": 37.074, "timestamp": "2026-02-04 03:46:07", "rationale": "AMD earnings after market close today. High volatility expected with options implying significant move. Aggressive pre-earnings position to capture potential gap up/down. AI chip demand remains strong narrative."}, {"symbol": "PYPL", "quantity": 50, "price": 54.108, "timestamp": "2026-02-04 03:46:14", "rationale": "PYPL down 20% after earnings miss and new CEO announcement. Oversold condition

### Now look at the trace

https://platform.openai.com/traces

### How many tools did we use in total?

In [11]:
from mcp_params import trader_mcp_server_params, researcher_mcp_server_params

all_params = trader_mcp_server_params + researcher_mcp_server_params("Faze")

count = 0
for each_params in all_params:
    async with MCPServerStdio(params=each_params, client_session_timeout_seconds=60) as server:
        mcp_tools = await server.list_tools()
        count += len(mcp_tools)
print(f"We have {len(all_params)} MCP servers, and {count} tools")

We have 5 MCP servers, and 15 tools
